In [ ]:
import numpy as np
import pandas as pd
import re
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.metrics import silhouette_score
from itertools import product
import torch
from bertopic.vectorizers import ClassTfidfTransformer



In [2]:
clean_articles_df = pd.read_parquet("clean_articles_step1.parquet")
clean_articles_df.shape
# clean_articles_df = pd.read_parquet("clean_articles_step1_(1).parquet")
# clean_articles_df.shape

(153512, 4)

In [3]:
clean_articles_df.head()

,article_id,date,title,content
0,1,2023-11-10,The Road Ahead: How China's AI Foundation Mode...,The Road Ahead: How China's AI Foundation Mode...
1,2,2023-11-19,Microsoft and Nvidia to Empower Developers wit...,Microsoft and Nvidia to Empower Developers wit...
2,3,2023-12-12,Google Releases New Chatbot Bard as a Strong C...,Google Releases New Chatbot Bard as a Strong C...
3,4,2023-09-07,Zoom Expands AI Offering with AI Companion and...,Zoom Expands AI Offering with AI Companion and...
4,5,2023-08-04,Pro-AI Thinking: Enhancing Industrial Environm...,Pro-AI Thinking: Enhancing Industrial Environm...


In [4]:
boilerplate_patterns = [
    r"skip\s*to\s*content\w*",
    r"watch\s*live\w*",
    r"community\s*calendar\w*",
    r"closings?(?:\s*&\s*|\s+and\s+)?delays?\b",
    r"submit\s*photos?(?:\s*&\s*|\s+and\s+)?videos?\b",
    r"download\s*(?:our\s*)?apps?\b",
    r"sign\s*up\s*for\s*(?:our\s*)?newsletters?\b",
    r"newsletters?\b",
    r"advertis\w*(?:\s+with\s+us)?\b",
    r"send\s*us\s*a\s*news\s*tip\b",
    r"weather\s*radar\b",
    r"election\s*results\b",
    r"home\s*news\s*weather\s*sports(?:\s*\w+){0,20}",
]
bp_regex = re.compile("|".join(boilerplate_patterns), flags=re.IGNORECASE)

def normalize_space(s: str) -> str:
    return re.sub(r"\s+", " ", str(s)).strip()

def topic_clean(title: str, content: str) -> str:
    t = normalize_space(title)
    c = normalize_space(content)

    # 去正文开头重复标题（最多两次）
    for _ in range(2):
        if t and c.lower().startswith(t.lower()):
            c = c[len(t):].strip(" -:|")

    # 删固定模板短语
    c = bp_regex.sub(" ", c)

    # 删导航词连续堆叠片段
    c = re.sub(
        r"(?:\b(home|news|weather|sports|live|video|search|menu|about|contact)\b[\s|/:,-]*){5,}",
        " ",
        c,
        flags=re.IGNORECASE,
    )

    c = re.sub(r"[\r\n\t]+", " ", c)
    c = re.sub(r"\s{2,}", " ", c).strip()
    return c

work_df = clean_articles_df.copy()
work_df["title"] = work_df["title"].fillna("").astype(str).str.strip()
work_df["content"] = work_df["content"].fillna("").astype(str).str.strip()

work_df["content_topic"] = [
    topic_clean(t, c) for t, c in zip(work_df["title"], work_df["content"])
]
work_df["text_raw"] = (work_df["title"] + " " + work_df["content_topic"]).str.strip()
work_df = work_df[work_df["text_raw"].str.len() > 0].reset_index(drop=True)

print("usable docs:", len(work_df))


# check = work_df["text_raw"].str.lower()
# for p in ["skip to content", "watch live", "community calendar", "advertise", "newsletter"]:
#     print(p, (check.str.contains(re.escape(p), regex=True)).mean())


usable docs: 153512


In [5]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(model_name)
tokenizer = embedder.tokenizer

MAX_TOKENS = 256  # test 先用 256

def truncate_text(text, max_tokens=256):
    ids = tokenizer.encode(
        text,
        add_special_tokens=True,
        truncation=True,
        max_length=max_tokens
    )
    return tokenizer.decode(ids, skip_special_tokens=True)

work_df["text_trunc"] = work_df["text_raw"].map(lambda x: truncate_text(x, MAX_TOKENS))


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
TEST_N = 10000
RANDOM_SEED = 42

if len(work_df) > TEST_N:
    test_df = work_df.sample(n=TEST_N, random_state=RANDOM_SEED).reset_index(drop=True)
else:
    test_df = work_df.copy()

docs_test = test_df["text_trunc"].tolist()
print("test docs:", len(docs_test))


test docs: 10000


In [7]:
# emb_test = embedder.encode(
#     docs_test,
#     batch_size=128,
#     show_progress_bar=True,
#     convert_to_numpy=True,
#     normalize_embeddings=True
# )

# print("embedding shape:", emb_test.shape)


In [8]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

embedder = SentenceTransformer(model_name, device=device)
emb_test = embedder.encode(
    docs_test,
    batch_size=256 if device in ["cuda", "mps"] else 128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("device:", device)
print("embedding shape:", emb_test.shape)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/40 [00:00<?, ?it/s]

device: mps
embedding shape: (10000, 384)


In [30]:
def eval_run(topics, emb):
    labels = np.array(topics)
    outlier_rate = np.mean(labels == -1)
    valid = labels != -1
    n_topics = len(set(labels[valid])) if valid.any() else 0

    sil = np.nan
    if valid.sum() > 20 and n_topics > 1:
        sil = silhouette_score(emb[valid], labels[valid])
        
    outlier_penalty = abs(outlier_rate - 0.225) * 2 if outlier_rate > 0.3 else abs(outlier_rate - 0.225)
    combined_score = (sil * 0.6) - outlier_penalty if not np.isnan(sil) else -outlier_penalty

    return {
        "n_topics": n_topics,
        "outlier_rate": float(outlier_rate),
        "silhouette": float(sil) if not np.isnan(sil) else np.nan,
        "combined_score": float(combined_score) if not np.isnan(combined_score) else np.nan
    }

param_grid = {
    "n_neighbors": [30, 35, 40, 45],         
    "min_dist": [0.05, 0.1, 0.15],          
    "min_cluster_size": [20, 25, 30],       
    "min_samples": [3, 5, 8]                
}

results = []
vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b"
)

total_runs = len(param_grid["n_neighbors"]) * len(param_grid["min_dist"]) * len(param_grid["min_cluster_size"]) * len(param_grid["min_samples"])
run_id = 0


for n_neighbors, min_dist, min_cluster_size, min_samples in product(
    param_grid["n_neighbors"],
    param_grid["min_dist"],
    param_grid["min_cluster_size"],
    param_grid["min_samples"]
):
    run_id += 1
    print(f"\n[{run_id}/{total_runs}] start -> nn={n_neighbors}, md={min_dist}, mcs={min_cluster_size}, ms={min_samples}")

    umap_model = UMAP(
        n_neighbors=n_neighbors,
        n_components=5,
        min_dist=min_dist,
        metric="cosine",
        random_state=42,
        low_memory=True
    )
    hdbscan_model = HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=False
    )

    tm = BERTopic(
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        calculate_probabilities=False,
        verbose=False
    )

    # topics, _ = tm.fit_transform(docs_test, emb_test)
    # m = eval_run(topics, emb_test)
    # m.update({
    #     "n_neighbors": n_neighbors,
    #     "min_dist": min_dist,
    #     "min_cluster_size": min_cluster_size,
    #     "min_samples": min_samples
    # })
    # results.append(m)

    try:
        topics, _ = tm.fit_transform(docs_test, emb_test)
        m = eval_run(topics, emb_test)
        m.update({
            "n_neighbors": n_neighbors,
            "min_dist": min_dist,
            "min_cluster_size": min_cluster_size,
            "min_samples": min_samples
        })
        results.append(m)
        print(f"[{run_id}/{total_runs}] done  -> topics={m['n_topics']}, outlier={m['outlier_rate']:.4f}, sil={m['silhouette']:.4f}")
    except Exception as e:
        print(f"[{run_id}/{total_runs}] skip  -> {type(e).__name__}: {e}")
        continue


res_df = pd.DataFrame(results)

res_df['topic_range_score'] = res_df['n_topics'].apply(lambda x: 1 if 50 <= x <= 80 else 0.5 if 30 <= x < 50 or 80 < x <= 100 else 0)

res_df = res_df.sort_values(
    by=["combined_score", "topic_range_score", "silhouette"], 
    ascending=[False, False, False]
)





[1/108] start -> nn=30, md=0.05, mcs=20, ms=3
[1/108] done  -> topics=109, outlier=0.3716, sil=0.0472

[2/108] start -> nn=30, md=0.05, mcs=20, ms=5
[2/108] done  -> topics=109, outlier=0.4065, sil=0.0512

[3/108] start -> nn=30, md=0.05, mcs=20, ms=8
[3/108] done  -> topics=100, outlier=0.4374, sil=0.0563

[4/108] start -> nn=30, md=0.05, mcs=25, ms=3
[4/108] done  -> topics=91, outlier=0.3921, sil=0.0477

[5/108] start -> nn=30, md=0.05, mcs=25, ms=5
[5/108] done  -> topics=83, outlier=0.4105, sil=0.0489

[6/108] start -> nn=30, md=0.05, mcs=25, ms=8
[6/108] done  -> topics=71, outlier=0.3660, sil=0.0428

[7/108] start -> nn=30, md=0.05, mcs=30, ms=3
[7/108] done  -> topics=71, outlier=0.3828, sil=0.0423

[8/108] start -> nn=30, md=0.05, mcs=30, ms=5
[8/108] done  -> topics=70, outlier=0.4052, sil=0.0445

[9/108] start -> nn=30, md=0.05, mcs=30, ms=8
[9/108] done  -> topics=59, outlier=0.3745, sil=0.0430

[10/108] start -> nn=30, md=0.1, mcs=20, ms=3
[10/108] done  -> topics=106, ou

In [32]:
res_df.head(20)

,n_topics,outlier_rate,silhouette,combined_score,n_neighbors,min_dist,min_cluster_size,min_samples,topic_range_score
62,47,0.3302,0.025463,-0.195122,40,0.05,30,8,0.5
79,3,0.0124,-0.007744,-0.217246,40,0.15,30,5,0.0
24,3,0.0118,-0.007521,-0.217713,30,0.15,30,3,0.0
25,3,0.0117,-0.007495,-0.217797,30,0.15,30,5,0.0
106,3,0.0079,-0.006463,-0.220978,45,0.15,30,5,0.0
87,66,0.3578,0.045858,-0.238085,45,0.05,30,3,1.0
84,78,0.3600,0.046426,-0.242144,45,0.05,25,3,1.0
98,44,0.3554,0.024653,-0.246008,45,0.10,30,8,0.5
5,71,0.3660,0.042814,-0.256311,30,0.05,25,8,1.0
76,56,0.3634,0.025039,-0.261777,40,0.15,25,5,1.0


In [33]:
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.vectorizers import ClassTfidfTransformer

vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b"
)
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

umap_model = UMAP(
    n_neighbors=40,
    n_components=5,
    min_dist=0.05,
    metric="cosine",
    random_state=42,
    low_memory=True
)

hdbscan_model = HDBSCAN(
    min_cluster_size=30,
    min_samples=8,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=False
)

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    calculate_probabilities=False,
    verbose=True
)

topics, _ = topic_model.fit_transform(docs_test, emb_test)
topic_info = topic_model.get_topic_info()
display(topic_info.head(20))


2026-03-08 01:15:29,558 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-08 01:15:45,477 - BERTopic - Dimensionality - Completed ✓
2026-03-08 01:15:45,478 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-08 01:15:45,638 - BERTopic - Cluster - Completed ✓
2026-03-08 01:15:45,644 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-08 01:15:46,784 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation,Representative_Docs
0,-1,3302,-1_google_openai_artificial_cookies,"[google, openai, artificial, cookies, icon, ar...","[as ai rewrites the content story, who owns th..."
1,0,2839,0_weather_alert_local_dc,"[weather, alert, local, dc, music, school, cou...",[insider q & a : openai cto mira murati on she...
2,1,251,1_transportation_electronic_news releases_busi...,"[transportation, electronic, news releases, bu...",[artificial intelligence applications become a...
3,2,237,2_india_delhi_indian_modi,"[india, delhi, indian, modi, minister, bengalu...",[us defence focuses on talent pipeline to attr...
4,3,210,3_venturebeat_automation_cloud_sap,"[venturebeat, automation, cloud, sap, aws, sec...",[the case for bringing generative ai on premis...
5,4,207,4_newswires_ein presswire_presswire_ein,"[newswires, ein presswire, presswire, ein, int...",[people to confide in ai chatbots for healthca...
6,5,153,5_oil energy_europe arab_arab_mena,"[oil energy, europe arab, arab, mena, asia afr...",[amatrium announces gpt - 5. 1 upgrade and ai ...
7,6,140,6_chatgpt_gpt_openai_chatbot,"[chatgpt, gpt, openai, chatbot, users, use cha...",[chatgpt will soon be able to see your mac app...
8,7,139,7_nvidia_gpus_chip_chips,"[nvidia, gpus, chip, chips, gpu, pc, hpc, amd,...",[tesla targets ai data centers with massive me...
9,8,139,8_screener_activity_commodities_market activity,"[screener, activity, commodities, market activ...",[2 artificial intelligence stocks to buy hand ...


In [37]:
doc_info = topic_model.get_document_info(docs_test)
display(doc_info[["Document", "Topic", "Name", "Probability"]].head(20))

# 看某个topic的样本文档
t = -1
sample_docs = doc_info[doc_info["Topic"] == t]["Document"].head(10).tolist()
for i, d in enumerate(sample_docs, 1):
    print(f"\n--- doc {i} ---\n{d[:500]}")


,Document,Topic,Name,Probability
0,warren buffett has bet over $ 156 billion on t...,8,8_screener_activity_commodities_market activity,1.000000
1,hoot launches ai integration on its myopia man...,0,0_weather_alert_local_dc,1.000000
2,twitch resurrects ai jesus - israel365 news se...,-1,-1_google_openai_artificial_cookies,0.000000
3,chatgpt passed the google level 3 entry test :...,6,6_chatgpt_gpt_openai_chatbot,1.000000
4,edjx and zeblok computational partner to deliv...,0,0_weather_alert_local_dc,1.000000
5,vodafone strikes billion - dollar ai deal with...,33,33_wireless_telco_ran_open ran,0.948275
6,artificial intelligence in healthcare market 2...,-1,-1_google_openai_artificial_cookies,0.000000
7,how ai can help take the emotion out of invest...,-1,-1_google_openai_artificial_cookies,0.000000
8,"eros innovation forays into aividya, launches ...",5,5_oil energy_europe arab_arab_mena,1.000000
9,eight us newspapers have sued openai and micro...,-1,-1_google_openai_artificial_cookies,0.000000



--- doc 1 ---
twitch resurrects ai jesus - israel365 news see all results subscribe idf biblical news israel news jerusalem christian zionism opinion menu idf biblical news israel news jerusalem christian zionism opinion study the bible donate to israel365 twitch resurrects ai jesus their idols are silver and gold, the work of men ' s hands. psalms 115 : 4 ( the israel bible ) adam eliyahu berkowitz medical / science june 18, 2023 2 min read home » twitch resurrects ai jesus jesus technology a new chatbot des

--- doc 2 ---
artificial intelligence in healthcare market 2022 : revenue growth, key factors, major companies, forecast to 2026 with dominant regions and countries data – tactical market skip to the content search tactical marketnews menu energy space environment technology satellites electric vehicles contact search search for : close search close menu energy space environment technology satellites electric vehicles contact categories uncategorized artificial intelligence in h

In [43]:
fig = topic_model.visualize_barchart(top_n_topics=46)
fig.show()

In [36]:
fig = topic_model.visualize_topics()
fig.show()

# Tag

In [44]:
# 0) 先拿到你现有结果（你已经有这些变量）
# test_df: 含 article_id, text_trunc
# topics: fit_transform 得到的 topic 列表
# topic_model: BERTopic 模型对象

doc_df = test_df[["article_id", "text_trunc"]].copy()
doc_df["topic_id"] = topics

In [46]:
# 1) 生成 topic 关键词表（基于你当前 BERTopic 结果）

topic_info = topic_model.get_topic_info().copy()
topic_info = topic_info[topic_info["Topic"] != -1]  # 去掉离群类
display(topic_info[["Topic", "Name", "Count"]].head(30))


,Topic,Name,Count
1,0,0_weather_alert_local_dc,2839
2,1,1_transportation_electronic_news releases_busi...,251
3,2,2_india_delhi_indian_modi,237
4,3,3_venturebeat_automation_cloud_sap,210
5,4,4_newswires_ein presswire_presswire_ein,207
6,5,5_oil energy_europe arab_arab_mena,153
7,6,6_chatgpt_gpt_openai_chatbot,140
8,7,7_nvidia_gpus_chip_chips,139
9,8,8_screener_activity_commodities_market activity,139
10,9,9_sportspremier leaguebasketballcombat_policyh...,131


In [47]:
noise_topics = [0,4,9,14,21,37,39,41]

doc_df = doc_df[~doc_df["topic_id"].isin(noise_topics)].copy()

In [48]:
# 2) 建立“topic -> 三维标签”映射（先手工填一版，后面可迭代）
# 你根据上面 topic_info 的 Name/关键词来填

topic_map = pd.DataFrame({

"topic_id":[
1,2,3,5,6,7,8,10,12,13,15,16,17,18,19,20,22,23,25,26,27,28,29,30,31,32,33,34,35,38,40,42,43,44,45
],

"industry":[
"Technology",        #1 enterprise tech
"Government",        #2 india politics
"Enterprise Software",#3 venturebeat cloud
"Energy",            #5 oil energy
"Technology",        #6 chatgpt
"Semiconductors",    #7 nvidia
"Finance",           #8 market analysis
"Government",        #10 federal policy
"Media",             #12 ai art
"Technology",        #13 china deepseek
"Marketing",         #15 crm marketing
"Consumer Electronics",#16 laptops
"Finance",           #17 crypto
"Technology",        #18 altman/openai
"Technology",        #19 machine learning
"Finance",           #20 nasdaq
"Government",        #22 trump
"Government",        #23 africa
"Legal",             #25 law
"Technology",        #26 gemini
"Finance",           #27 stocks
"Cybersecurity",     #28 security
"Real Estate",       #29 realty
"Technology",        #30 musk/xai
"Finance",           #31 etfs
"Consumer Electronics",#32 apple
"Telecommunications",#33 telco
"Technology",        #34 claude
"Finance",           #35 investment
"Labor Market",      #38 data scientist
"Government",        #40 EU AI act
"Media",             #42 ai images
"Media",             #43 ai music
"Semiconductors",    #44 nvidia
"Finance"            #45 trading
],

"technology":[
"Enterprise AI",
"Other",
"Cloud / Automation",
"Other",
"LLM / GenAI",
"AI Infrastructure",
"Predictive Analytics",
"Other",
"Generative Media",
"LLM / GenAI",
"AI Agents",
"Recommendation Systems",
"Blockchain",
"LLM / GenAI",
"Machine Learning",
"Predictive Analytics",
"Other",
"Other",
"Legal AI",
"LLM / GenAI",
"Predictive Analytics",
"Cybersecurity",
"Predictive Analytics",
"LLM / GenAI",
"Predictive Analytics",
"Generative AI",
"Network AI",
"Code Generation",
"Predictive Analytics",
"Other",
"Other",
"Image Generation",
"Generative Audio",
"AI Infrastructure",
"AI Infrastructure"
],

"factor":[
"Adoption",
"Policy",
"Enterprise Adoption",
"Other",
"Adoption",
"Infrastructure",
"Market Impact",
"Regulation",
"Copyright",
"Geopolitics",
"Efficiency",
"Consumer Adoption",
"Regulation",
"Leadership",
"Innovation",
"Market",
"Policy",
"Digital Access",
"Regulation",
"Competition",
"Market",
"Security Risk",
"Market",
"Competition",
"Market",
"Consumer Adoption",
"Infrastructure",
"Productivity",
"Investment",
"Talent",
"Regulation",
"Copyright",
"Copyright",
"Infrastructure",
"Market"
]

})



In [49]:
# 3) merge 到文章级（这是你要的结构化主表）
article_tags = doc_df.merge(topic_map, on="topic_id", how="left")

# 未映射 topic 先补 Other
for c in ["industry", "technology", "factor"]:
    article_tags[c] = article_tags[c].fillna("Other")

display(article_tags.head())


,article_id,text_trunc,topic_id,industry,technology,factor
0,49883,warren buffett has bet over $ 156 billion on t...,8,Finance,Predictive Analytics,Market Impact
1,85346,twitch resurrects ai jesus - israel365 news se...,-1,Other,Other,Other
2,141545,chatgpt passed the google level 3 entry test :...,6,Technology,LLM / GenAI,Adoption
3,3431,vodafone strikes billion - dollar ai deal with...,33,Telecommunications,Network AI,Infrastructure
4,120894,artificial intelligence in healthcare market 2...,-1,Other,Other,Other


In [ ]:

print("Industry distribution")
display(article_tags["industry"].value_counts())

print("Technology distribution")
display(article_tags["technology"].value_counts())

print("Factor distribution")
display(article_tags["factor"].value_counts())

Industry distribution


industry
Other                   3561
Technology               797
Government               531
Finance                  475
Enterprise Software      210
Media                    175
Semiconductors           170
Energy                   153
Consumer Electronics     131
Marketing                 88
Legal                     65
Cybersecurity             58
Real Estate               56
Telecommunications        46
Labor Market              37
Name: count, dtype: int64

Technology distribution


technology
Other                     4282
LLM / GenAI                424
Predictive Analytics       422
Enterprise AI              251
Cloud / Automation         210
AI Infrastructure          201
Generative Media           113
AI Agents                   88
Recommendation Systems      81
Blockchain                  78
Machine Learning            77
Legal AI                    65
Cybersecurity               58
Generative AI               50
Network AI                  46
Code Generation             45
Image Generation            31
Generative Audio            31
Name: count, dtype: int64

Factor distribution


factor
Other                  3714
Adoption                391
Policy                  305
Regulation              303
Market                  269
Infrastructure          216
Enterprise Adoption     210
Copyright               175
Market Impact           139
Consumer Adoption       131
Competition             114
Geopolitics              93
Efficiency               88
Innovation               77
Leadership               77
Digital Access           66
Security Risk            58
Productivity             45
Investment               45
Talent                   37
Name: count, dtype: int64

In [51]:
pd.crosstab(article_tags["industry"], article_tags["technology"])
pd.crosstab(article_tags["industry"], article_tags["factor"])
pd.crosstab(article_tags["technology"], article_tags["factor"])


factor,Adoption,Competition,Consumer Adoption,Copyright,Digital Access,Efficiency,Enterprise Adoption,Geopolitics,Infrastructure,Innovation,Investment,Leadership,Market,Market Impact,Other,Policy,Productivity,Regulation,Security Risk,Talent
technology,,,,,,,,,,,,,,,,,,,,
AI Agents,0,0,0,0,0,88,0,0,0,0,0,0,0,0,0,0,0,0,0,0
AI Infrastructure,0,0,0,0,0,0,0,0,170,0,0,0,31,0,0,0,0,0,0,0
Blockchain,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,78,0,0
Cloud / Automation,0,0,0,0,0,0,210,0,0,0,0,0,0,0,0,0,0,0,0,0
Code Generation,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,45,0,0,0
Cybersecurity,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,58,0
Enterprise AI,251,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Generative AI,0,0,50,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Generative Audio,0,0,0,31,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
